In [19]:
import os
import glob
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
from pathlib import Path


DATA_DAILY = Path("data/daily")
DATA_HOT_SCORE = Path("data/hotscore")
DATA_RECOMMENDATIONS = Path("data/recommendations")
OUTPUT_RECOMENDATIONS = Path("output/recommendations")

for p in (DATA_DAILY, DATA_HOT_SCORE, DATA_RECOMMENDATIONS, OUTPUT_RECOMENDATIONS):
    p.mkdir(parents=True, exist_ok=True)

In [20]:
# -------------------------
# Load daily hot stocks
# -------------------------
files = sorted(glob.glob(str(DATA_DAILY / "hot_stocks_*.csv")))
if not files:
    raise FileNotFoundError(f"No hot stocks files found in {DATA_DAILY}")

dfs = []

for i, f in enumerate(files):
    df = pd.read_csv(f)

    if 'symbol' not in df.columns and 'Symbol' in df.columns:
        df = df.rename(columns={'Symbol':'symbol'})
        
    dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True, sort=False)
full_df.head()



,symbol,regularMarketPrice,regularMarketChangePercent,regularMarketVolume,averageDailyVolume3Month,marketCap,VolumeSpike,MomentumScore,VolumeScore,VolatilityScore,TrendScore,HotScore
0,DAVE,306.355,6.825794,1099029.0,574007.0,3.895073e+09,1.914661,0.813397,0.964115,0.933014,0.959330,0.904665
1,MRVL,311.290,11.294242,44084905.0,34943947.0,2.725510e+11,1.261589,0.961722,0.885167,0.976077,0.550239,0.896651
2,ENTG,163.020,8.304544,3682585.0,2792768.0,2.482795e+10,1.318615,0.901914,0.897129,0.875598,0.825359,0.887321
3,OMAB,108.940,7.045304,170231.0,84719.0,5.258662e+09,2.009360,0.830144,0.968900,0.796651,0.978469,0.886842
4,WDC,652.530,15.917758,8013917.0,7761507.0,2.249155e+11,1.032521,0.997608,0.717703,0.995215,0.851675,0.884569


In [21]:

# Ensure numeric columns
cols_num = [
    'regularMarketPrice',
    'regularMarketChangePercent',
    'regularMarketVolume',
    'averageDailyVolume3Month',
    'marketCap',
    'HotScore',
    'VolumeSpike',
    'MomentumScore',
    'VolumeScore',
    'VolatilityScore',
    'TrendScore'
]

for c in cols_num:
    if c in full_df.columns:
        full_df[c] = pd.to_numeric(full_df[c], errors='coerce')


keep_cols = [
    'symbol',
    'HotScore',
    'TrendScore',
    'regularMarketPrice',
    'regularMarketChangePercent',
    'VolumeSpike',
    'averageDailyVolume3Month',
    'MomentumScore',
    'VolumeScore',
    'VolatilityScore',
    'marketCap'
]


for c in keep_cols:
    if c not in full_df.columns:
        full_df[c] = np.nan

full_df = full_df[keep_cols]
full_df['symbol'] = full_df['symbol'].astype(str).str.upper()
full_df = full_df.sort_values(['HotScore','symbol']).reset_index(drop=True)



# -------------------------
# Save HotScore snapshot
# -------------------------


timestamp = datetime.now().strftime("%Y%m%d")
hotscore_file = DATA_HOT_SCORE / f"hotscore_{timestamp}.csv"

if hotscore_file.exists():
    old_df = pd.read_csv(hotscore_file)
    full_df = pd.concat([old_df, full_df], ignore_index=True)

full_df.to_csv(hotscore_file, index=False)
full_df.head(4)


,symbol,HotScore,TrendScore,regularMarketPrice,regularMarketChangePercent,VolumeSpike,averageDailyVolume3Month,MomentumScore,VolumeScore,VolatilityScore,marketCap
0,MOD,0.556977,0.891473,296.0750,3.624181,0.144343,1133971.0,0.658915,0.155039,0.914729,1.563743e+10
1,ESAB,0.559690,0.829457,101.4750,4.916250,0.117997,706357.0,0.844961,0.100775,0.728682,6.177972e+09
2,SVM,0.563178,0.403101,12.6303,3.695612,0.268572,4006531.0,0.666667,0.689922,0.240310,2.793854e+09
3,STEP,0.565116,0.705426,49.1800,4.527101,0.163842,1070028.0,0.821705,0.294574,0.519380,6.198456e+09


In [22]:
import plotly.express as px

#df_hotscore = full_df.groupby('symbol')['HotScore'].mean().sort_values(ascending=True).reset_index()

df_hotscore = (
    full_df.groupby('symbol')
    .agg({
        'HotScore': 'mean',
        'regularMarketPrice': 'last'
    })
    .reset_index()
    .sort_values('HotScore', ascending=False)
    .head(40)
    .sort_values('HotScore')
)

fig = px.bar(
    df_hotscore.head(40),
    x='HotScore',
    y='symbol',
    color="HotScore",
    text="regularMarketPrice",
    orientation="h",
    color_continuous_scale='ice',
)

fig.update_traces(textposition="inside", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title='Top Symbols by Average HotScore',     
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=1200
)

fig.show()

chart_path = os.path.join(OUTPUT_RECOMENDATIONS, "bar_avg_hot_score.html")
fig.write_html(chart_path, include_plotlyjs="cdn")

In [23]:
stats = full_df.sort_index().groupby('symbol').last().reset_index()

fig = px.bar(
    stats.sort_values(by=["regularMarketPrice", "HotScore"], ascending=True).head(40),
    x="HotScore", 
    y="symbol",
    orientation="h",
    color="regularMarketPrice",
    text="regularMarketPrice",
    hover_data=["TrendScore","VolatilityScore"], 
    title="Average Hot Score vs Average Volume Spike",
    color_continuous_scale="Blues",
)

fig.update_traces(textposition="inside", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title="Top 40 Average Hot Score Stocks",
    xaxis_title="Hot Score",
    yaxis_title="STOCKS",    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=1200,
    margin=dict(l=40, r=40, t=60, b=40)
)
    
fig.show()


chart_path = os.path.join(OUTPUT_RECOMENDATIONS, "bar_hot_score.html")
fig.write_html(chart_path, include_plotlyjs="cdn")

In [24]:
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
avg_csv = DATA_RECOMMENDATIONS / f"trade_suggestions_{timestamp}.csv"
stats.to_csv(avg_csv) 


In [25]:
stats.head()

,symbol,HotScore,TrendScore,regularMarketPrice,regularMarketChangePercent,VolumeSpike,averageDailyVolume3Month,MomentumScore,VolumeScore,VolatilityScore,marketCap
0,AAL,0.566667,0.093023,16.125,4.302815,0.238109,70078392.0,0.790698,0.620155,0.317829,1.066484e+10
1,AAOI,0.837201,0.641148,191.710,13.404321,0.989493,12738271.0,0.990431,0.677033,0.947368,1.538334e+10
2,ACIW,0.865152,0.833333,45.170,5.242310,1.877374,788839.0,0.931818,0.926768,0.656566,4.592053e+09
3,ACLS,0.797129,0.935407,191.415,6.270818,1.078202,690953.0,0.760766,0.755981,0.863636,5.882929e+09
4,ACMR,0.827811,0.902367,96.190,5.494628,1.358797,1426331.0,0.869822,0.757396,0.840237,6.647642e+09


In [26]:
top = (
    full_df.groupby('symbol')
    .agg({
        'HotScore': 'mean',         
        'marketCap': 'last',      
        'regularMarketPrice': 'last'
    })
    .reset_index()
)
top.head() 

,symbol,HotScore,marketCap,regularMarketPrice
0,AAL,0.566667,1.066484e+10,16.125
1,AAOI,0.837201,1.538334e+10,191.710
2,ACIW,0.865152,4.592053e+09,45.170
3,ACLS,0.797129,5.882929e+09,191.415
4,ACMR,0.827811,6.647642e+09,96.190


In [27]:
fig = px.scatter(
    top.sort_values(by=["regularMarketPrice", "HotScore"], ascending=True).head(50),
    x='marketCap',
    y='HotScore',
    size='regularMarketPrice',
    text='regularMarketPrice',
    color='HotScore',
    hover_name='symbol',
    log_x=True,
    title='HotScore vs Market Cap',
    color_continuous_scale="ice",
)

fig.update_traces(textposition="top right", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title="Top 50 Averages Hot Score Stocks", 
    xaxis_title="Market Cap",
    yaxis_title="Hot Score",    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=800,
    margin=dict(l=40, r=40, t=60, b=40)
)


fig.show()


chart_path = os.path.join(OUTPUT_RECOMENDATIONS, "scatter_avg_hot_score.html")
fig.write_html(chart_path, include_plotlyjs="cdn")